<a href="https://colab.research.google.com/github/domysolano/Working-With-The-Pandas-Library/blob/main/WorkingWithThePandasLibrary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Maestría en Inteligencia Artificial y Analítica de Datos**

Curso: *Programación para Analítica Descriptiva y Predictiva*

* Semestre: Enero-Junio 2026
* Profesor: Dr. Vicente García Jiménez
* Alumno: Ricardo Solano Monje
* Matrícula: 266221
* Unidad 1
* Práctica 6: Manejo de librería Pandas.
* Objetivo de la práctica: Usar Pandas para manipular un dataset predefinido, realizando operaciones de análisis de datos sobre éste.
* Realizado por: Ricardo Solano Monje

## Before beginning, let's know about Titanic dataset metadata

The dataset includes 891 rows and 12 columns, containing demographic and trip information.

The dataset is frequently split into train.csv (891 rows) and test.csv (418 rows), with the latter lacking survival information.

**Missing Data**: **Age** and Cabin often contain significant missing values requiring imputation.


The columns present in the Titanic dataset are:

*PassengerId:* This column contains a unique identifier for each passenger.

**Survived**: This column contains information about whether a passenger survived the sinking or not. The value 0 indicates that the passenger did not survive, while the value 1 indicates that the passenger did survive.

**Pclass**: This column contains information about the passenger’s ticket class. The value 1 indicates first class and so on (1 = 1st/Upper, 2 = 2nd/Middle, 3 = 3rd/Lower).

*Name*: This column contains the name of the passenger.

**Sex**: This column contains information about the passenger’s gender, (male/female).

**Age**: This column contains the age of the passenger, (fractional if less than 1).

**SibSp**: This column contains information about the passenger’s siblings and spouse aboard.

**Parch**: This column contains information about the passenger’s parents and children aboard.

*Ticket*: This column contains the ticket number of the passenger.

**Fare**: This column contains the fare paid by the passenger.

*Cabin*: This column contains the cabin number of the passenger.

*Embarked*: This column contains information about the port of embarkation,C=Cherbourg, Q=Queenstown, S=Southampton).
    

## Práctica 6 Manejo de librería Pandas

# Instrucciones:

*Carga el archivo titanic.csv en la carpeta correspondiente de Google drive **(/content/drive/MyDrive/ClassFiles/Titanic-Dataset.csv)** para realizar los siguientes ejercicios:*

* **Ejercicio 1**: Análisis de la distribución de supervivencia por combinación de sexo y clase del pasajero.

  * Calcula la proporción de supervivencia para cada combinación de 'Sex' y 'Pclass'.
  * Identifica qué combinación tuvo la tasa de supervivencia más alta.
  * Identifica qué combinación tuvo la tasa de supervivencia más baja.

* **Ejercicio 2**: Identificación de familias grandes a bordo.

  * Crea una nueva columna 'FamilySize' sumando las columnas 'SibSp' y 'Parch'.
  * Considera como "familia grande" a aquellas donde 'FamilySize' es mayor a 3.
  * Calcula el número de pasajeros en familias grandes.
  * Calcula la proporción de supervivencia entre los pasajeros que pertenecen a familias grandes.

* **Ejercicio 3**: Segmentación por grupos de edad.

  * Clasifica a los pasajeros en las siguientes categorías de edad(tip puede resultar mas sencillo realizarlo con una función):  menor de edad (< 18) y mayor de edad (>=18)

* **Ejercicio 4**: Comparación entre promedios calculados manualmente y con Pandas

  * Utiliza NumPy para calcular el promedio de las columnas 'Age' y 'Fare', ignorando valores nulos.
  * Compara estos valores con los promedios obtenidos utilizando los métodos nativos de Pandas.
  * Verifica que los resultados sean consistentes.

* **Ejercicio 5**. Creación de intervalos de clase usando NumPy y análisis con Pandas

  * Divide la columna 'Fare' en 5 intervalos equidistantes utilizando la función numpy.linspace, el estudiante deberá investigar la operación de esta función en python.

  * Crea una nueva columna en el DataFrame que asigne a cada pasajero el intervalo correspondiente de su tarifa.

  * Calcula el número de pasajeros en cada intervalo utilizando Pandas y la proporción de supervivientes por intervalo.


### Ejercicio 1: Análisis de la distribución de supervivencia por combinación de sexo y clase del pasajero.

* Calcula la proporción de supervivencia para cada combinación de 'Sex' y 'Pclass'.
* Identifica qué combinación tuvo la tasa de supervivencia más alta.
* Identifica qué combinación tuvo la tasa de supervivencia más baja.

In [90]:
from google.colab import drive
import os

# if the /content/drive path is available then NOT MOUNT it

def mount_drive_if_needed():
    if not os.path.isdir('/content/drive'):
        drive.mount('/content/drive')

In [91]:
# Import libraries
import pandas as pd
import numpy as np

def load_and_explore_data(filePath):
    """
    Load the Titanic dataset and perform initial exploration.

    Parameters:
    filepath (str): Path to the CSV file

    Returns:
    pd.DataFrame: Loaded Titanic data
    """
    # Load data from CSV file
    df = pd.read_csv(filePath)
    """
    # Basic dataset exploration
    print("=" * 60)
    print("DATASET EXPLORATION")
    print("=" * 60)

    # Display basic information
    print(f"1. Dataset has {df.shape[0]} rows and {df.shape[1]} columns")
    print(f"2. Column names: {list(df.columns)}")
    print("\n3. First 5 rows:")
    print(df.head())

    # Check data types and missing values
    print("\n4. Data Types and Missing Values:")
    print(df.info())
    """
    return df


def calculate_survival_by_group(df, group_columns, target_column='Survived'):
    """
    Calculate survival statistics for grouped data.

    Parameters:
    df (pd.DataFrame): Input DataFrame
    group_columns (list): Columns to group by (e.g., ['Sex', 'Pclass'])
    target_column (str): Column indicating survival (0/1)

    Returns:
    pd.DataFrame: Survival statistics for each group
    """
    # Group the data and calculate survival statistics
    grouped_stats = df.groupby(group_columns)[target_column].agg([
        # 'count' counts all rows in each group
        ('total_passengers', 'count'),

        # 'sum' adds up all 1s (survivors) in each group
        # Since Survived=1 means survived, Survived=0 means died
        ('survived_count', 'sum'),

        # Calculate survival rate as percentage
        # lambda function: for each group x, calculate (survivors/total)*100
        ('survival_rate', lambda x: (x.sum() / x.count() * 100).round(2))
    ]).reset_index()  # Convert grouped result to regular DataFrame

    return grouped_stats


def identify_extreme_survival_rates(stats_df):
    """
    Identify combinations with highest and lowest survival rates.

    Parameters:
    stats_df (pd.DataFrame): DataFrame with survival statistics

    Returns:
    tuple: (highest_row, lowest_row) as Series objects
    """
    # Find index of row with maximum survival rate
    max_idx = stats_df['survival_rate'].idxmax()
    highest_survival = stats_df.loc[max_idx]

    # Find index of row with minimum survival rate
    min_idx = stats_df['survival_rate'].idxmin()
    lowest_survival = stats_df.loc[min_idx]

    return highest_survival, lowest_survival


def display_survival_analysis(stats_df, highest_row, lowest_row):
    """
    Display formatted results of survival analysis.

    Parameters:
    stats_df (pd.DataFrame): Survival statistics DataFrame
    highest_row (pd.Series): Row with highest survival rate
    lowest_row (pd.Series): Row with lowest survival rate
    """
    print("\n" + "=" * 60)
    print("SURVIVAL ANALYSIS BY SEX AND CLASS")
    print("=" * 60)

    # Display all survival rates
    print("\n" + "-" * 60)
    print("1. SURVIVAL RATE FOR EACH SEX AND CLASS COMBINATION:")
    print("-" * 60)
    print(stats_df.to_string(index=False))

    # Display highest survival rate
    print("\n" + "-" * 60)
    print("2. COMBINATION WITH HIGHEST SURVIVAL RATE:")
    print("-" * 60)
    print(f"   Sex: {highest_row['Sex']}")
    print(f"   Passenger Class: {highest_row['Pclass']}")
    print(f"   Survival Rate: {highest_row['survival_rate']}%")
    print(f"   Survived: {int(highest_row['survived_count'])} out of {int(highest_row['total_passengers'])} passengers")

    # Display lowest survival rate
    print("\n" + "-" * 60)
    print("3. COMBINATION WITH LOWEST SURVIVAL RATE:")
    print("-" * 60)
    print(f"   Sex: {lowest_row['Sex']}")
    print(f"   Passenger Class: {lowest_row['Pclass']}")
    print(f"   Survival Rate: {lowest_row['survival_rate']}%")
    print(f"   Survived: {int(lowest_row['survived_count'])} out of {int(lowest_row['total_passengers'])} passengers")






def main():
    """
    Main function to execute the complete analysis.
    """
    # File path, txt file is in ClassFiles folder as requested.
    file_path = "/content/drive/MyDrive/ClassFiles/Titanic-Dataset.csv"
    mount_drive_if_needed()
    # Step 1: Load and explore the data
    df = load_and_explore_data(file_path)

    # Step 2: Calculate survival statistics by sex and class
    survival_stats = calculate_survival_by_group(df, ['Sex', 'Pclass'])

    # Step 3: Identify highest and lowest survival rates
    highest, lowest = identify_extreme_survival_rates(survival_stats)

    # Step 4: Display the analysis results
    display_survival_analysis(survival_stats, highest, lowest)

    toBeAcomplish="""
    Ejercicio 1: Análisis de la distribución de supervivencia por combinación de sexo y clase del pasajero.
    Calcula la proporción de supervivencia para cada combinación de 'Sex' y 'Pclass'.
    Identifica qué combinación tuvo la tasa de supervivencia más alta.
    Identifica qué combinación tuvo la tasa de supervivencia más baja.
    """
    print(toBeAcomplish)

# Run the analysis
if __name__ == "__main__":
    main()


SURVIVAL ANALYSIS BY SEX AND CLASS

------------------------------------------------------------
1. SURVIVAL RATE FOR EACH SEX AND CLASS COMBINATION:
------------------------------------------------------------
   Sex  Pclass  total_passengers  survived_count  survival_rate
female       1                94              91          96.81
female       2                76              70          92.11
female       3               144              72          50.00
  male       1               122              45          36.89
  male       2               108              17          15.74
  male       3               347              47          13.54

------------------------------------------------------------
2. COMBINATION WITH HIGHEST SURVIVAL RATE:
------------------------------------------------------------
   Sex: female
   Passenger Class: 1
   Survival Rate: 96.81%
   Survived: 91 out of 94 passengers

------------------------------------------------------------
3. COMBINATIO

## Ejercicio 2: Identificación de familias grandes a bordo.

  * Crea una nueva columna 'FamilySize' sumando las columnas 'SibSp' y 'Parch'.
  * Considera como "familia grande" a aquellas donde 'FamilySize' es mayor a 3.
  * Calcula el número de pasajeros en familias grandes.
  * Calcula la proporción de supervivencia entre los pasajeros que pertenecen a familias grandes.

In [92]:
import pandas as pd
import numpy as np

def load_and_display_data(filePath):
    """
    Load the Titanic dataset and display basic information.

    Returns:
    pd.DataFrame: Loaded Titanic data
    """

    # Load data from CSV file
    df = pd.read_csv(filePath)

    return df


def create_family_size_column(df):
    """
    Create a new 'FamilySize' column by adding 'SibSp' and 'Parch'.

    Parameters:
    df (pd.DataFrame): Original Titanic data

    Returns:
    pd.DataFrame: DataFrame with new 'FamilySize' column
    """
    print("\n" + "="*70)
    print("STEP 1: CREATING 'FamilySize' COLUMN")
    print("="*70)

    # Display what SibSp and Parch mean
    print("Column definitions:")
    print("- SibSp: Number of siblings/spouses aboard")
    print("- Parch: Number of parents/children aboard")
    print("\n" + "="*70)
    print("\nExample values:")
    print(df[['Name', 'SibSp', 'Parch']].head())

    # Create FamilySize column by adding SibSp and Parch
    # We add 1 to include the passenger themselves
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

    print("\n" + "-"*70)
    print("\nAfter adding FamilySize column:")
    print(f"FamilySize = SibSp + Parch + 1 (the passenger themselves)")
    print(df[['Name', 'SibSp', 'Parch', 'FamilySize']].head(10))
    print("\n" + "-"*70)
    return df


def identify_large_families(df, threshold=3):
    """
    Identify large families based on FamilySize threshold.

    Parameters:
    df (pd.DataFrame): DataFrame with FamilySize column
    threshold (int): FamilySize value above which a family is considered large

    Returns:
    pd.DataFrame: DataFrame with families marked as large or not
    """
    print("\n" + "="*70)
    print("STEP 2: IDENTIFYING LARGE FAMILIES")
    print("="*70)

    # Create a boolean column to mark large families
    df['IsLargeFamily'] = df['FamilySize'] > threshold

    print(f"Definition: Large family = FamilySize > {threshold}")
    print("\nDistribution of FamilySize values:")
    family_size_counts = df['FamilySize'].value_counts().sort_index()
    print(family_size_counts)

    print(f"\nFamilies with size > {threshold}:")
    print(family_size_counts[family_size_counts.index > threshold])

    # Show examples of large families
    print("\nExamples of large families (>3 members):")
    large_families = df[df['IsLargeFamily'] == True]
    print(large_families[['Name', 'SibSp', 'Parch', 'FamilySize', 'Survived']].head(10))

    return df


def calculate_passengers_in_large_families(df):
    """
    Calculate the number of passengers in large families.

    Parameters:
    df (pd.DataFrame): DataFrame with family information

    Returns:
    int: Number of passengers in large families
    """
    print("\n" + "="*70)
    print("STEP 3: CALCULATING PASSENGERS IN LARGE FAMILIES")
    print("="*70)

    # Count passengers in large families
    passengers_in_large_families = df['IsLargeFamily'].sum()

    # Calculate percentage
    total_passengers = len(df)
    percentage = (passengers_in_large_families / total_passengers) * 100

    print(f"Total passengers on Titanic: {total_passengers}")
    print(f"Passengers in large families (FamilySize > 3): {passengers_in_large_families}")
    print(f"Percentage: {percentage:.2f}% of all passengers")

    return passengers_in_large_families


def calculate_survival_rate_large_families(df):
    """
    Calculate survival rate among passengers in large families.

    Parameters:
    df (pd.DataFrame): DataFrame with family and survival information

    Returns:
    float: Survival rate for large family passengers
    """
    print("\n" + "="*70)
    print("STEP 4: CALCULATING SURVIVAL RATE FOR LARGE FAMILIES")
    print("="*70)

    # Filter for large families only
    large_families_df = df[df['IsLargeFamily'] == True]

    if len(large_families_df) == 0:
        print("No large families found!")
        return 0

    # Calculate survival rate
    total_large_family_passengers = len(large_families_df)
    survived_large_family_passengers = large_families_df['Survived'].sum()

    survival_rate = (survived_large_family_passengers / total_large_family_passengers) * 100

    print(f"Large family passengers analyzed: {total_large_family_passengers}")
    print(f"Large family passengers who survived: {survived_large_family_passengers}")
    print(f"Survival rate for large families: {survival_rate:.2f}%")

    return survival_rate


def main():
    """
    Main function to execute the complete analysis.
    """
    # File path, txt file is in ClassFiles folder as requested.
    filePath = "/content/drive/MyDrive/ClassFiles/Titanic-Dataset.csv"
    mount_drive_if_needed()

    try:
        # Step 1: Load and display data
        df = load_and_display_data(filePath)# Load the Titanic dataset
        # Step 2: Create FamilySize column
        df = create_family_size_column(df)

        # Step 3: Identify large families
        df = identify_large_families(df, threshold=3)

        # Step 4: Calculate number of passengers in large families
        large_family_passengers = calculate_passengers_in_large_families(df)

        # Step 5: Calculate survival rate for large families
        survival_rate = calculate_survival_rate_large_families(df)

        print("\n" + "="*70)
        print("Acomplished:")
        print(f"1. Created 'FamilySize' column (SibSp + Parch + 1)")
        print(f"2. Defined large families as FamilySize > 3")
        print(f"3. Found {large_family_passengers} passengers in large families")
        print(f"4. Survival rate for large families: {survival_rate:.2f}%")
        print("="*70)
    except FileNotFoundError:
        print("\nERROR: Could not find 'Titanic-Dataset.csv'")

    except Exception as e:
        print(f"\nERROR: {e}")
        print("Please check your data and try again.")
    toBeAcomplish="""
    Ejercicio 2: Identificación de familias grandes a bordo.
    Crea una nueva columna 'FamilySize' sumando las columnas 'SibSp' y 'Parch'.
    Considera como "familia grande" a aquellas donde 'FamilySize' es mayor a 3.
    Calcula el número de pasajeros en familias grandes.
    Calcula la proporción de supervivencia entre los pasajeros que pertenecen a familias grandes.
    """
    print(toBeAcomplish)


# Run the analysis
if __name__ == "__main__":
    main()


STEP 1: CREATING 'FamilySize' COLUMN
Column definitions:
- SibSp: Number of siblings/spouses aboard
- Parch: Number of parents/children aboard


Example values:
                                                Name  SibSp  Parch
0                            Braund, Mr. Owen Harris      1      0
1  Cumings, Mrs. John Bradley (Florence Briggs Th...      1      0
2                             Heikkinen, Miss. Laina      0      0
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)      1      0
4                           Allen, Mr. William Henry      0      0

----------------------------------------------------------------------

After adding FamilySize column:
FamilySize = SibSp + Parch + 1 (the passenger themselves)
                                                Name  SibSp  Parch  FamilySize
0                            Braund, Mr. Owen Harris      1      0           2
1  Cumings, Mrs. John Bradley (Florence Briggs Th...      1      0           2
2                             Heikki

## Ejercicio 3: Segmentación por grupos de edad.

Clasifica a los pasajeros en las siguientes categorías de edad(tip puede resultar mas sencillo realizarlo con una función):  menor de edad (< 18) y mayor de edad (>=18)

In [93]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def load_and_prepare_data(filePath):
    """
    Load Titanic dataset and prepare it for analysis.

    Parameters:
    filepath (str): Path to the CSV file

    Returns:
    pd.DataFrame: Prepared Titanic data
    """
    print("="*70)
    print("LOADING AND PREPARING DATA")
    print("="*70)

    # Load the dataset
    df = pd.read_csv(filePath)

    print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"\nAge column composition:")
    print(f"Unique age values: {df['Age'].nunique()}")
    print(f"Missing values: {df['Age'].isnull().sum()} ({df['Age'].isnull().sum()/len(df)*100:.1f}%)")



    return df

# STRATEGY FOR MISSING AGES: drop   - Remove rows with missing ages
def handle_missing_ages(df, strategy='drop'):
    """
    Handle missing age values with drop strategy.

    Parameters:
    df (pd.DataFrame): Titanic dataset
    strategy (str): How to handle missing ages ('median', 'mean', or 'drop'), but just drop was implemented.

    Returns:
    pd.DataFrame: DataFrame with handled missing ages
    """
    print("\n" + "="*70)
    print(f"HANDLING MISSING AGES (Strategy: {strategy})")
    print("="*70)

    original_missing = df['Age'].isnull().sum()
    original_rows = df.shape[0]

    if strategy == 'drop':
        rows_before = df.shape[0]
        df = df.dropna(subset=['Age'])
        rows_after = df.shape[0]
        rows_dropped = rows_before - rows_after

        print(f"  Dropped {rows_dropped} rows with missing ages")
        print(f"  Dataset changed: {rows_before} → {rows_after} rows")
        print(f"  Removed {rows_dropped/rows_before*100:.1f}% of data")

    else:
        print(f" Unknown strategy '{strategy}'. Keeping missing values.")
        print(f" Missing ages remain: {df['Age'].isnull().sum()}")

    print(f"\nMissing ages after handling: {df['Age'].isnull().sum()}")
    return df


def create_age_category_column(df, minor_threshold=18):
    """
    Create age category column based on threshold.

    Parameters:
    df (pd.DataFrame): DataFrame with Age column
    minor_threshold (int): Age threshold for minor/adult classification

    Returns:
    pd.DataFrame: DataFrame with new 'Age_Category' column
    """
    print("\n" + "="*70)
    print(f"CREATING AGE CATEGORIES (Minor: <{minor_threshold}, Adult: >={minor_threshold})")
    print("="*70)

    # Create age category using np.where
    df['Age_Category'] = np.where(
        df['Age'] < minor_threshold,
        'Minor',
        'Adult'
    )

    print("\nSample of passengers with age categories:")
    sample_df = df[['Name', 'Age', 'Age_Category']].head(10).copy()

    # Highlight minors in the sample
    def highlight_minor(row):
        if row['Age_Category'] == 'Minor':
            return ['background-color: yellow'] * 3
        return [''] * 3

    # Display with formatting
    print(sample_df.to_string(index=False))

    # Count categories
    minor_count = (df['Age_Category'] == 'Minor').sum()
    adult_count = (df['Age_Category'] == 'Adult').sum()

    print(f"\n  Created Age_Category column")
    print(f"  Minors (<{minor_threshold}): {minor_count} passengers")
    print(f"  Adults (≥{minor_threshold}): {adult_count} passengers")

    return df


def analyze_age_distribution(df):
    """
    Analyze and display age distribution statistics.

    Parameters:
    df (pd.DataFrame): DataFrame with Age and Age_Category columns

    Returns:
    pd.Series: Category counts
    """
    print("\n" + "="*70)
    print("AGE DISTRIBUTION ANALYSIS")
    print("="*70)

    # Count by category
    category_counts = df['Age_Category'].value_counts()
    category_percentages = df['Age_Category'].value_counts(normalize=True) * 100

    print("\nPassenger Count by Age Category:")
    for category in ['Minor', 'Adult']:
        count = category_counts.get(category, 0)
        percentage = category_percentages.get(category, 0)
        print(f"  {category}: {count:4d} passengers ({percentage:5.1f}%)")

    return category_counts




def main():
    """
    Main function to execute the complete age segmentation analysis.
    """
    try:
        # File path, txt file is in ClassFiles folder as requested.
        filePath = "/content/drive/MyDrive/ClassFiles/Titanic-Dataset.csv"
        mount_drive_if_needed()

        # Step 1: Load and prepare data
        df = load_and_prepare_data(filePath)

        #  set choose strategy as drop
        print("\n" + "-"*70)
        print("STRATEGY FOR MISSING AGES: DROP")

        print(" drop   - Remove rows with missing ages")
        print(" So, analysis is going to be based on subset of dataset")



        # Step 3: Handle missing age values
        df = handle_missing_ages(df, strategy='drop')

        # Step 4: Create age category column
        df = create_age_category_column(df, minor_threshold=18)

        # Step 5: Analyze age distribution
        category_counts = analyze_age_distribution(df)

        # Acomplished
        total_passengers = df.shape[0]
        minors = category_counts.get('Minor', 0)
        adults = category_counts.get('Adult', 0)
        print("\n" + "="*70)
        print(f"\nAcomplished:")
        print(f"Total passengers analyzed: {total_passengers}")
        print(f"Minors (<18 years): {minors} ({minors/total_passengers*100:.1f}%)")
        print(f"Adults (≥18 years): {adults} ({adults/total_passengers*100:.1f}%)")
        print(f"Strategy used: drop")
        print("="*70)

    except FileNotFoundError:
        print("\nERROR: Could not find 'Titanic-Dataset.csv'")

    except Exception as e:
        print(f"\nERROR: {e}")
        import traceback
        traceback.print_exc()
    toBeAcomplish="""
    Ejercicio 3: Segmentación por grupos de edad.
    Clasifica a los pasajeros en las siguientes categorías de edad: menor de edad (< 18) y mayor de edad (>=18)
    """
    print(toBeAcomplish)

# Run the analysis
if __name__ == "__main__":
    main()

LOADING AND PREPARING DATA
Dataset loaded: 891 rows, 12 columns

Age column composition:
Unique age values: 88
Missing values: 177 (19.9%)

----------------------------------------------------------------------
STRATEGY FOR MISSING AGES: DROP
 drop   - Remove rows with missing ages
 So, analysis is going to be based on subset of dataset

HANDLING MISSING AGES (Strategy: drop)
  Dropped 177 rows with missing ages
  Dataset changed: 891 → 714 rows
  Removed 19.9% of data

Missing ages after handling: 0

CREATING AGE CATEGORIES (Minor: <18, Adult: >=18)

Sample of passengers with age categories:
                                               Name  Age Age_Category
                            Braund, Mr. Owen Harris 22.0        Adult
Cumings, Mrs. John Bradley (Florence Briggs Thayer) 38.0        Adult
                             Heikkinen, Miss. Laina 26.0        Adult
       Futrelle, Mrs. Jacques Heath (Lily May Peel) 35.0        Adult
                           Allen, Mr. William Henr

## Ejercicio 4: Comparación entre promedios calculados manualmente y con Pandas

* Utiliza NumPy para calcular el promedio de las columnas 'Age' y 'Fare', ignorando valores nulos.
* Compara estos valores con los promedios obtenidos utilizando los métodos nativos de Pandas.
* Verifica que los resultados sean consistentes.

In [94]:
import pandas as pd
import numpy as np


def load_and_prepare_data(filePath):
    """
    Load Titanic dataset and prepare it for analysis.

    Parameters:
    filepath (str): Path to the CSV file

    Returns:
    pd.DataFrame: Prepared Titanic data
    """
    print("="*80)
    print("LOADING TITANIC DATASET FOR AVERAGE COMPARISON")
    print("="*80)

    # Load the dataset
    df = pd.read_csv(filePath)

    print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"\nColumns to analyze: Age and Fare")

    # Show basic information about the columns
    print("\nAge column:")
    print(f"  Missing values: {df['Age'].isnull().sum()} ({df['Age'].isnull().sum()/len(df)*100:.1f}%)")
    print(f"  Data type: {df['Age'].dtype}")

    print("\nFare column:")
    print(f"  Missing values: {df['Fare'].isnull().sum()} ({df['Fare'].isnull().sum()/len(df)*100:.1f}%)")
    print(f"  Data type: {df['Fare'].dtype}")

    """ # Show sample values
    print("\nSample values (first 5 rows):")
    print(df[['Age', 'Fare']].head())
    """
    return df


def calculate_numpy_averages(df):
    """
    Calculate averages using NumPy, manually handling missing values.

    Parameters:
    df (pd.DataFrame): Titanic dataset

    Returns:
    dict: Averages calculated with NumPy
    """
    print("\n" + "="*80)
    print("CALCULATING AVERAGES WITH NUMPY (Manual Approach)")
    print("="*80)

    results = {}

    # ============================================
    # COLUMN 1: AGE
    # ============================================
    print("\n1. CALCULATING AVERAGE AGE WITH NUMPY:")
    print("-"*40)

    # Get Age column values as NumPy array
    age_array = df['Age'].values
    print(f"  Raw Age array shape: {age_array.shape}")
    print(f"  First 10 values: {age_array[:10]}")

    # Step 1: Identify non-null values
    # np.isnan() checks for NaN (Not a Number) values
    age_not_nan = ~np.isnan(age_array)
    valid_age_count = np.sum(age_not_nan)
    print(f"\n  Step 1: Identify non-null values")
    print(f"    Total values: {len(age_array)}")
    print(f"    Valid (non-null) values: {valid_age_count}")
    print(f"    Missing values: {len(age_array) - valid_age_count}")

    # Step 2: Extract only valid values
    valid_ages = age_array[age_not_nan]
    print(f"\n  Step 2: Extract valid values")
    print(f"    Valid ages array shape: {valid_ages.shape}")
    print(f"    First 10 valid ages: {valid_ages[:10]}")

    # Step 3: Calculate mean manually
    # Mean = Sum of all values / Number of values
    age_sum = np.sum(valid_ages)
    age_mean_numpy = age_sum / valid_age_count
    print(f"\n  Step 3: Calculate mean")
    print(f"    Sum of all valid ages: {age_sum:.2f}")
    print(f"    Count of valid ages: {valid_age_count}")
    print(f"    Mean = {age_sum:.2f} / {valid_age_count} = {age_mean_numpy:.4f}")

    results['Age_NumPy'] = age_mean_numpy
    results['Age_Valid_Count'] = valid_age_count

    # ============================================
    # COLUMN 2: FARE
    # ============================================
    print("\n2. CALCULATING AVERAGE FARE WITH NUMPY:")
    print("-"*40)

    # Get Fare column values
    fare_array = df['Fare'].values
    print(f"  Raw Fare array shape: {fare_array.shape}")
    print(f"  First 10 values: {fare_array[:10]}")

    # Step 1: Identify non-null values
    # For Fare, we should also check for NaN
    fare_not_nan = ~np.isnan(fare_array)
    valid_fare_count = np.sum(fare_not_nan)
    print(f"\n  Step 1: Identify non-null values")
    print(f"    Total values: {len(fare_array)}")
    print(f"    Valid (non-null) values: {valid_fare_count}")
    print(f"    Missing values: {len(fare_array) - valid_fare_count}")

    # Step 2: Extract valid values
    valid_fares = fare_array[fare_not_nan]
    print(f"\n  Step 2: Extract valid values")
    print(f"    Valid fares array shape: {valid_fares.shape}")
    print(f"    First 10 valid fares: {valid_fares[:10]}")

    # Step 3: Calculate mean
    fare_sum = np.sum(valid_fares)
    fare_mean_numpy = fare_sum / valid_fare_count
    print(f"\n  Step 3: Calculate mean")
    print(f"    Sum of all valid fares: {fare_sum:.2f}")
    print(f"    Count of valid fares: {valid_fare_count}")
    print(f"    Mean = {fare_sum:.2f} / {valid_fare_count} = {fare_mean_numpy:.4f}")

    results['Fare_NumPy'] = fare_mean_numpy
    results['Fare_Valid_Count'] = valid_fare_count

    return results


def calculate_pandas_averages(df):
    """
    Calculate averages using Pandas built-in methods.

    Parameters:
    df (pd.DataFrame): Titanic dataset

    Returns:
    dict: Averages calculated with Pandas
    """
    print("\n" + "="*80)
    print("CALCULATING AVERAGES WITH PANDAS (Built-in Methods)")
    print("="*80)

    results = {}

    # ============================================
    # COLUMN 1: AGE
    # ============================================
    print("\n1. CALCULATING AVERAGE AGE WITH PANDAS:")
    print("-"*40)

    # Method 1: Using .mean() - automatically handles NaN
    age_mean_pandas = df['Age'].mean()
    print(f"  Using df['Age'].mean():")
    print(f"    Result: {age_mean_pandas:.4f}")
    print(f"    Note: .mean() automatically ignores NaN values")

    # Method 2: Using .describe() to get multiple statistics
    age_stats = df['Age'].describe()
    """
    print(f"\n  Using df['Age'].describe():")
    print(f"    Count (non-null): {age_stats['count']}")
    print(f"    Mean: {age_stats['mean']:.4f}")
    print(f"    Std:  {age_stats['std']:.4f}")
    print(f"    Min:  {age_stats['min']:.2f}")
    print(f"    25%:  {age_stats['25%']:.2f}")
    print(f"    50%:  {age_stats['50%']:.2f} (median)")
    print(f"    75%:  {age_stats['75%']:.2f}")
    print(f"    Max:  {age_stats['max']:.2f}")
    """
    results['Age_Pandas'] = age_mean_pandas
    results['Age_Count_Pandas'] = age_stats['count']

    # ============================================
    # COLUMN 2: FARE
    # ============================================
    print("\n2. CALCULATING AVERAGE FARE WITH PANDAS:")
    print("-"*40)

    # Method 1: Using .mean()
    fare_mean_pandas = df['Fare'].mean()
    print(f"  Using df['Fare'].mean():")
    print(f"    Result: {fare_mean_pandas:.4f}")

    # Method 2: Using .describe()
    fare_stats = df['Fare'].describe()
    """
    print(f"\n  Using df['Fare'].describe():")
    print(f"    Count (non-null): {fare_stats['count']}")
    print(f"    Mean: {fare_stats['mean']:.4f}")
    print(f"    Std:  {fare_stats['std']:.4f}")
    print(f"    Min:  {fare_stats['min']:.2f}")
    print(f"    25%:  {fare_stats['25%']:.2f}")
    print(f"    50%:  {fare_stats['50%']:.2f} (median)")
    print(f"    75%:  {fare_stats['75%']:.2f}")
    print(f"    Max:  {fare_stats['max']:.2f}")
    """
    results['Fare_Pandas'] = fare_mean_pandas
    results['Fare_Count_Pandas'] = fare_stats['count']

    return results


def compare_results(numpy_results, pandas_results):
    """
    Compare results from NumPy and Pandas calculations.

    Parameters:
    numpy_results (dict): Results from NumPy calculations
    pandas_results (dict): Results from Pandas calculations
    """
    print("\n" + "="*80)
    print("COMPARING NUMPY vs PANDAS RESULTS")
    print("="*80)

    # Create comparison table
    comparison_data = []

    # Compare Age
    age_numpy = numpy_results['Age_NumPy']
    age_pandas = pandas_results['Age_Pandas']
    age_diff = abs(age_numpy - age_pandas)

    comparison_data.append({
        'Column': 'Age',
        'NumPy Mean': f"{age_numpy:.6f}",
        'Pandas Mean': f"{age_pandas:.6f}",
        'Difference': f"{age_diff:.10f}",
        'Match?': ' YES' if age_diff < 0.000001 else ' NO',
        'NumPy Count': numpy_results['Age_Valid_Count'],
        'Pandas Count': pandas_results['Age_Count_Pandas']
    })

    # Compare Fare
    fare_numpy = numpy_results['Fare_NumPy']
    fare_pandas = pandas_results['Fare_Pandas']
    fare_diff = abs(fare_numpy - fare_pandas)

    comparison_data.append({
        'Column': 'Fare',
        'NumPy Mean': f"{fare_numpy:.6f}",
        'Pandas Mean': f"{fare_pandas:.6f}",
        'Difference': f"{fare_diff:.10f}",
        'Match?': ' YES' if fare_diff < 0.000001 else ' NO',
        'NumPy Count': numpy_results['Fare_Valid_Count'],
        'Pandas Count': pandas_results['Fare_Count_Pandas']
    })

    # Create and display DataFrame
    comparison_df = pd.DataFrame(comparison_data)
    print("\nComparison Table:")
    print(comparison_df.to_string(index=False))

     # Check if means match to prove consistency
    print("\nChecking for consistency:")
    print("MEAN VALUES")
    print(f"   Age difference:  {age_diff:.10f}")
    print(f"   Fare difference: {fare_diff:.10f}")

    if age_diff < 0.000001 and fare_diff < 0.000001:
        print(" SUCCESS: NumPy and Pandas results are CONSISTENT!")
        #print("  The tiny differences are due to floating-point precision.")
    else:
        print(" WARNING: Significant differences detected!")
        print("  Please check your calculations.")


def main():
    """
    Main function to execute the comparison analysis.
    """
    try:
         # File path, txt file is in ClassFiles folder as requested.
        filePath = "/content/drive/MyDrive/ClassFiles/Titanic-Dataset.csv"
        mount_drive_if_needed()

        # Step 1: Load and prepare data
        df = load_and_prepare_data(filePath)

        # Step 2: Calculate averages with NumPy (manual approach)
        numpy_results = calculate_numpy_averages(df)

        # Step 3: Calculate averages with Pandas (built-in methods)
        pandas_results = calculate_pandas_averages(df)

        # Step 4: Compare results
        compare_results(numpy_results, pandas_results)


        print("\n" + "="*80)
        print("SUMMARY AND ACOMPLISHMENTS")
        print("1. Both methods should produce identical results")
        print("2. Very small differences are due to floating-point precision.")
        print("3. Results are consistent.")

    except FileNotFoundError:
        print("\nERROR: Could not find 'Titanic-Dataset.csv'")
    except Exception as e:
        print(f"\nERROR: {e}")
        import traceback
        traceback.print_exc()
    toBeAcomplish="""
    Ejercicio 4: Comparación entre promedios calculados manualmente y con Pandas
    Utiliza NumPy para calcular el promedio de las columnas 'Age' y 'Fare', ignorando valores nulos.
    Compara estos valores con los promedios obtenidos utilizando los métodos nativos de Pandas.
    Verifica que los resultados sean consistentes.
    """
    print(toBeAcomplish)


# Run the analysis
if __name__ == "__main__":
    main()

LOADING TITANIC DATASET FOR AVERAGE COMPARISON
Dataset loaded: 891 rows, 12 columns

Columns to analyze: Age and Fare

Age column:
  Missing values: 177 (19.9%)
  Data type: float64

Fare column:
  Missing values: 0 (0.0%)
  Data type: float64

CALCULATING AVERAGES WITH NUMPY (Manual Approach)

1. CALCULATING AVERAGE AGE WITH NUMPY:
----------------------------------------
  Raw Age array shape: (891,)
  First 10 values: [22. 38. 26. 35. 35. nan 54.  2. 27. 14.]

  Step 1: Identify non-null values
    Total values: 891
    Valid (non-null) values: 714
    Missing values: 177

  Step 2: Extract valid values
    Valid ages array shape: (714,)
    First 10 valid ages: [22. 38. 26. 35. 35. 54.  2. 27. 14.  4.]

  Step 3: Calculate mean
    Sum of all valid ages: 21205.17
    Count of valid ages: 714
    Mean = 21205.17 / 714 = 29.6991

2. CALCULATING AVERAGE FARE WITH NUMPY:
----------------------------------------
  Raw Fare array shape: (891,)
  First 10 values: [ 7.25   71.2833  7.925  

## Ejercicio 5. Creación de intervalos de clase usando NumPy y análisis con Pandas

* Divide la columna 'Fare' en 5 intervalos equidistantes utilizando la función numpy.linspace.

* Crea una nueva columna en el DataFrame que asigne a cada pasajero el intervalo correspondiente de su tarifa.
  
* Calcula el número de pasajeros en cada intervalo utilizando Pandas y la proporción de supervivientes por intervalo.

Note: There was a FutureWarning while testing the code, for a concise explanation check function *calculate_passengers_and_survivors(df)*

In [95]:
import pandas as pd
import numpy as np

def load_and_prepare_data(filePath):
    """
    Load Titanic dataset and prepare it for analysis.

    Parameters:
    filepath (str): Path to the CSV file

    Returns:
    pd.DataFrame: Prepared Titanic data
    """
    print("="*80)
    print("LOADING TITANIC DATASET FOR  CREATING INTERVALS\n")
    print("="*80)

    # Load the dataset
    df = pd.read_csv(filePath)


    print("Dataset loaded successfully.\n")
    print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

    return df


def create_fare_intervals(fare_column, num_intervals=5):
    """
    Create equidistant intervals for fare values using numpy.linspace.

    Parameters:
    fare_column (pd.Series): The Fare column from Titanic dataset
    num_intervals (int): Number of intervals to create (default: 5)

    Returns:
    np.array: Array of interval boundaries
    """
    print("\n" + "="*60)
    print(f"CREATING {num_intervals} EQUIDISTANT FARE INTERVALS") # (Divide la columna 'Fare' en 5 intervalos equidistantes)
    print("="*60)

    # Get min and max fare values
    min_fare = fare_column.min()
    max_fare = fare_column.max()

    print(f"Minimum fare: £{min_fare:.2f}")
    print(f"Maximum fare: £{max_fare:.2f}")
    print(f"Range: £{max_fare - min_fare:.2f}")

    # Create equidistant intervals using numpy.linspace
    # linspace creates num_intervals+1 points that divide the range equally
    intervals = np.linspace(start=min_fare, stop=max_fare, num=num_intervals + 1)

    print(f"\nCreated {len(intervals)} boundary points, equivalent to {len(intervals)-1} intervals :")
    for i, boundary in enumerate(intervals):
        print(f"  Boundary {i}: £{boundary:.2f}")

    return intervals


def assign_passengers_to_intervals(df, intervals):
    """
    Assign each passenger to a fare interval and create a new column.

    Parameters:
    df (pd.DataFrame): Titanic dataset
    intervals (np.array): Array of interval boundaries

    Returns:
    pd.DataFrame: DataFrame with new 'Fare_Interval' column
    """
    print("\n" + "="*60)
    print("New column:--> Fare_Interval")
    print("ASSIGNING PASSENGERS TO THE NEW COLUMN FARE INTERVALS") # asigne a cada pasajero el intervalo correspondiente de su tarifa.
    print("="*60)

    # We need to create interval labels (e.g., "£0.00-£102.47"), i.e. names for intervales
    interval_labels = []
    for i in range(len(intervals) - 1):
        label = f"£{intervals[i]:.2f}-£{intervals[i+1]:.2f}"
        interval_labels.append(label)

    print(f"We have created {len(interval_labels)} interval labels:")
    for i, label in enumerate(interval_labels):
        print(f"  Interval {i+1}: {label}")

    # Use pandas cut function to assign each fare value to an interval
    # bins=intervals defines the boundaries
    # labels=interval_labels assigns , it  names  each interval
    # include_lowest=True ensures the minimum value is included in first interval
    # Crea una nueva columna en el DataFrame que asigne a cada pasajero el intervalo correspondiente de su tarifa.

    df['Fare_Interval'] = pd.cut(
        x=df['Fare'],          # Column to bin
        bins=intervals,        # Interval boundaries
        labels=interval_labels,# Names for intervals
        include_lowest=True    # Include the minimum value
    )

    # Show first 10 passengers with their fare intervals
    print("\nFirst 10 passengers with fare intervals TO WHERE THEY WERE ASSIGNED:")
    print(df[['Name', 'Fare', 'Fare_Interval']].head(10).to_string(index=False))

    return df


def analyze_interval_distribution(df):
    """
    Calculate number and proportion of passengers in each fare interval.

    Parameters:
    df (pd.DataFrame): DataFrame with 'Fare_Interval' column
    """


    # Calculate number of passengers in each interval
    # value_counts() counts occurrences of each interval
    # sort_index() sorts by interval (maintains fare order)
    interval_counts = df['Fare_Interval'].value_counts().sort_index()

    # Calculate total number of passengers
    total_passengers = len(df)

    # Calculate proportion (percentage) in each interval
    interval_proportions = (interval_counts / total_passengers * 100).round(2)

    print("\n" + "="*60)
    print("ANALYZING PASSENGER DISTRIBUTION AND SURVIVORS BY FARE INTERVAL")

    print(f"\nTotal passengers analyzed: {total_passengers}")
    print(f"Number of intervals: {len(interval_counts)}")
    print("="*60)

    # Create a summary DataFrame
    summary_df = pd.DataFrame({
        'Interval': interval_counts.index,
        'Passenger_Count': interval_counts.values,
     #   'Proportion_Percent': interval_proportions.values
    })

    print("\nPassenger Distribution by Fare Interval:") # distribución de pasajeros en cada intervalo
    print(summary_df.to_string(index=False))

    return summary_df

def calculate_passengers_and_survivors(df):
    """
    Calculate:
    1. Number of passengers in each interval
    2. Proportion of survivors in each interval
    """
    # Group by Fare_Interval and calculate required statistics

    """There was a FutureWarning while testing the code, specifically while testing group by, this is the reason and the solution (at least temporally)
    This FutureWarning occurs because pandas is changing how groupby handles categorical data, defaulting to only including observed categories.
    To fix it, add observed=True or observed=False inside the groupby() call to explicitly define whether
    to show all possible categorical levels or just those present in your data, effectively silencing the warning.
    To adopt the future behavior (only show categories present in data use: result = df.groupby('Fare_Interval', observed=True).agg(...)
    OR to keep the current behavior (show all categories even if empty use: result = df.groupby('Fare_Interval', observed=False).agg(...)

    I have choosen observed=False,
    """

    result = df.groupby('Fare_Interval', observed=False).agg(
        Passenger_Count=('Survived', 'count'),      # Count all passengers in interval
        Survivor_Count=('Survived', 'sum'),         # Count survivors (Survived=1)
        Survival_Proportion=('Survived', 'mean')    # Mean = survivors/total = proportion
    ).reset_index()
    # Print results
    print("\n" + "="*70)
    print("PASSENGER DISTRIBUTION AND SURVIVORS BY FARE INTERVAL:") # Pasajeros en cada intervalo y proporción de supervivientes por intervalo
    print("=" * 70)
    result['Survival_Proportion']=round(result['Survival_Proportion']*100,2)
    print(result.to_string(index=False))
    return result


def main():
    """
    Main function to execute the complete analysis.
    """
    try:
        # File path, txt file is in ClassFiles folder as requested.
        filePath = "/content/drive/MyDrive/ClassFiles/Titanic-Dataset.csv"
        mount_drive_if_needed()

        # Step 1: Load and prepare the Titanic dataset
        df = load_and_prepare_data(filePath)


        # Step 2: Create 5 equidistant fare intervals using numpy.linspace
        intervals = create_fare_intervals(df['Fare'], num_intervals=5)

        # Crea una nueva columna en el DataFrame que asigne a cada pasajero el intervalo correspondiente de su tarifa.
        # Step 3: Create a new column and assign passengers to intervals
        df = assign_passengers_to_intervals(df, intervals)

        # Step 4: Calculate number and proportion of passengers in each interval
        summary_df = analyze_interval_distribution(df)
          # 4. Calculate passengers and survivor proportion per interval
        result = calculate_passengers_and_survivors(df)
        # Print results
        #print("\nFare Interval Analysis: (pasajeros en cada intervalo y proporción de supervivientes)")
        #print("=" * 70)
        #print(result.to_string(index=False))

        print("\n" + "="*60)
        print("DONE")
        print("="*60)
        print("\nRequested Tasks completed:")
        print("1. Created 5 equidistant fare intervals using numpy.linspace")
        print("2. Added new 'Fare_Interval' column to DataFrame")
        print("3. Calculated passenger count and proportion for each interval")

    except FileNotFoundError:
        print("ERROR: Could not find 'titanic.csv' file.")
        print("Please ensure the file is in the current directory.")
    except Exception as e:
        print(f"ERROR: {e}")
    toBeAcomplish="""
    Ejercicio 5. Creación de intervalos de clase usando NumPy y análisis con Pandas
    1. Divide la columna 'Fare' en 5 intervalos equidistantes utilizando la función numpy.linspace.
    2. Crea una nueva columna en el DataFrame que asigne a cada pasajero el intervalo correspondiente de su tarifa.
    3. Calcula el número de pasajeros en cada intervalo utilizando Pandas y la proporción de supervivientes por intervalo.
    """
    print(toBeAcomplish)

# Execute the main function
if __name__ == "__main__":
    main()

LOADING TITANIC DATASET FOR  CREATING INTERVALS

Dataset loaded successfully.

Shape: 891 rows, 12 columns

CREATING 5 EQUIDISTANT FARE INTERVALS
Minimum fare: £0.00
Maximum fare: £512.33
Range: £512.33

Created 6 boundary points, equivalent to 5 intervals :
  Boundary 0: £0.00
  Boundary 1: £102.47
  Boundary 2: £204.93
  Boundary 3: £307.40
  Boundary 4: £409.86
  Boundary 5: £512.33

New column:--> Fare_Interval
ASSIGNING PASSENGERS TO THE NEW COLUMN FARE INTERVALS
We have created 5 interval labels:
  Interval 1: £0.00-£102.47
  Interval 2: £102.47-£204.93
  Interval 3: £204.93-£307.40
  Interval 4: £307.40-£409.86
  Interval 5: £409.86-£512.33

First 10 passengers with fare intervals TO WHERE THEY WERE ASSIGNED:
                                               Name    Fare Fare_Interval
                            Braund, Mr. Owen Harris  7.2500 £0.00-£102.47
Cumings, Mrs. John Bradley (Florence Briggs Thayer) 71.2833 £0.00-£102.47
                             Heikkinen, Miss. Laina 